# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JawadKhan65/glamourrr/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import userdata
try:
    import datasets
except ImportError:
    !pip install datasets -q
    import datasets

# Using the HF_TOKEN from your secrets
hf_token = userdata.get('HF_TOKEN')
dataset_name = "FlyRank/internship-warehouse"
config_name = "fact_content_daily_performance"

In [6]:
import pandas as pd
import numpy as np
from google.colab import userdata
import datasets

hf_token = userdata.get('HF_TOKEN')
dataset_name = "FlyRank/internship-warehouse"
config_name = "fact_content_daily_performance"

try:
    print(f"Streaming data for config '{config_name}' with unique IDs...")
    ds = datasets.load_dataset(dataset_name, config_name, token=hf_token, streaming=True)

    data_list = []
    max_rows = 20000
    records_scanned = 0
    max_scan = 150000

    iterable = ds['train'].iter(batch_size=1000)

    for batch in iterable:
        records_scanned += len(batch['report_date'])
        dates = [str(d) for d in batch['report_date']]

        for i, date_str in enumerate(dates):
            # Look for March data (allowing 2025 as the latest available year in this slice)
            if date_str.startswith('2026-03') or date_str.startswith('2025-03'):
                data_list.append({
                    'date': date_str,
                    'client_id': batch.get('client_hash_id', ['unknown']*len(dates))[i],
                    'content_id': batch.get('content_hash_id', ['unknown']*len(dates))[i],
                    'device': batch.get('device', ['unknown']*len(dates))[i],
                    'position': batch.get('gsc_avg_position', [0]*len(dates))[i],
                    'impressions': batch.get('gsc_impressions', [0]*len(dates))[i],
                    'clicks': batch.get('gsc_clicks', [0]*len(dates))[i]
                })

            if len(data_list) >= max_rows:
                break
        if len(data_list) >= max_rows or records_scanned >= max_scan:
            break

    df_sample = pd.DataFrame(data_list)

    if not df_sample.empty:
        print(f"✅ Loaded {len(df_sample):,} rows. Scanned {records_scanned:,} records.")

        # Updated Verification
        print("\n--- Verification ---")
        # New grain includes the hash IDs
        grain_key = ['date', 'client_id', 'content_id', 'device']
        max_dupes = df_sample.groupby(grain_key).size().max()

        print(f"Fact 1 (Grain): {max_dupes == 1} (Max per grain: {max_dupes})")
        print(f"Fact 2 (Span): {df_sample['date'].min()} to {df_sample['date'].max()}")

        df_sample['is_valid_signal'] = (df_sample['position'] > 0) & (df_sample['impressions'] > 0)
        survivors = df_sample[df_sample['is_valid_signal'] == True]
        print(f"Fact 3 (Availability): {len(survivors):,} rows survive ({(len(survivors)/len(df_sample)):.1%})")

        display(df_sample.head(3))
    else:
        print(f"⚠ No March data found in {records_scanned:,} records.")

except Exception as e:
    print(f"❌ Execution failed: {e}")

Streaming data for config 'fact_content_daily_performance' with unique IDs...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

✅ Loaded 20,000 rows. Scanned 98,000 records.

--- Verification ---
Fact 1 (Grain): True (Max per grain: 1)
Fact 2 (Span): 2025-03-01 to 2025-03-21
Fact 3 (Availability): 19,966 rows survive (99.8%)


,date,client_id,content_id,device,position,impressions,clicks,is_valid_signal
0,2025-03-01,client_73cda7b4e4f265ea,content_30744f0df03fae4d,unknown,65.5,2,0,True
1,2025-03-01,client_73cda7b4e4f265ea,content_eb59ee4f00886795,unknown,74.0,6,0,True
2,2025-03-01,client_73cda7b4e4f265ea,content_e487f64cb6f0bf09,unknown,72.0,1,0,True


In [20]:
# 4. Build Feature Frame & Leakage Trap Experiment
try:
    # Updated to use client_id and content_id instead of query/landing_page
    df_features = df_sample[['date', 'client_id', 'content_id', 'device', 'position']].copy()
    df_features['day_of_week'] = pd.to_datetime(df_features['date']).dt.dayofweek

    print("--- Leakage Trap Experiment ---")
    # STEP A: Inject the 'Trap' (Label Leakage)
    df_features['LEAK_clicks'] = df_sample['clicks']
    print("Step A: Added 'LEAK_clicks' to features. This would cause 100% accuracy (Leakage).")
    display(df_features[['client_id', 'position', 'LEAK_clicks']].head(3))

    # STEP B: The Cure (Delete the leaked column)
    df_features = df_features.drop(columns=['LEAK_clicks'])
    print("\nStep B: Deleted 'LEAK_clicks'. The feature frame is now honest and ready for modeling.")
    display(df_features.head(3))
except NameError:
    print("Please run the data loading cell successfully first.")

--- Leakage Trap Experiment ---
Step A: Added 'LEAK_clicks' to features. This would cause 100% accuracy (Leakage).


,client_id,position,LEAK_clicks
0,client_73cda7b4e4f265ea,65.5,0
1,client_73cda7b4e4f265ea,74.0,0
2,client_73cda7b4e4f265ea,72.0,0



Step B: Deleted 'LEAK_clicks'. The feature frame is now honest and ready for modeling.


,date,client_id,content_id,device,position,day_of_week
0,2025-03-01,client_73cda7b4e4f265ea,content_30744f0df03fae4d,unknown,65.5,5
1,2025-03-01,client_73cda7b4e4f265ea,content_eb59ee4f00886795,unknown,74.0,5
2,2025-03-01,client_73cda7b4e4f265ea,content_e487f64cb6f0bf09,unknown,72.0,5


## 1. Unit of analysis + time window

**The Contract (Search Intelligence Lane):**
1. **One row means:** A single (Query, Landing Page, Device) combination for a specific date.
2. **Tables used:** `FlyRank/internship-warehouse` (Search Console + Analytics sync).
3. **Time window:** Mid-panel month `2026-03` for development.
4. **Prediction/Rank:** Probability of a `click` (Binary proxy: clicks > 0).
5. **Deliberately exclude:** I exclude the `impressions` count from the feature set to avoid hindsight bias (if we have many impressions, we already know the search happened).

In [13]:
try:
    # Fact 1: The Grain (Verification)
    # Using the IDs that we confirmed provide a unique grain
    grain_key = ['date', 'client_id', 'content_id', 'device']
    grain_check = df_sample.groupby(grain_key).size().max()
    print(f"Fact 1: Max rows per unique grain key: {grain_check} (Success: {grain_check == 1})")

    # Fact 2: Slice Row Count and Date Span
    print(f"Fact 2: Row count for mid-panel: {len(df_sample):,}")
    print(f"Date span: {df_sample['date'].min()} to {df_sample['date'].max()}")
except NameError:
    print("Please run the data loading cell (c944867f) first.")

Fact 1: Max rows per unique grain key: 1 (Success: True)
Fact 2: Row count for mid-panel: 20,000
Date span: 2025-03-01 to 2025-03-21



## 2. Fields: feature / label / context / excluded

- **Features:** `query`, `landing_page`, `device`, `position` (previous day avg).
- **Label:** `is_clicked` (Derived from `clicks > 0`).
- **Context:** `date`, `country`.
- **Excluded:** `revenue` (Excluded because it is a downstream outcome, not a ranking signal knowable at the request moment).

In [15]:
try:
    # Fact 3: Availability (Filter with IS TRUE)
    # Using pre-calculated is_valid_signal from the loading cell
    survivors = df_sample[df_sample['is_valid_signal'] == True]
    print(f"Fact 3: {len(survivors):,} rows survive availability filter.")
    print(f"Survival rate: {len(survivors)/len(df_sample):.1%}")
except Exception as e:
    print(f"Check failed: {e}")

Fact 3: 19,966 rows survive availability filter.
Survival rate: 99.8%


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
try:
    # --- The Five Feature Frame ---
    # Updated columns to match current df_sample schema
    df_features = df_sample[['date', 'client_id', 'content_id', 'device', 'position']].copy()
    df_features['day_of_week'] = pd.to_datetime(df_features['date']).dt.dayofweek

    # --- THE TRAP (Leakage) ---
    df_features['LEAK_clicks'] = df_sample['clicks']
    print("CRITICAL: Feature frame with LEAKAGE (The Trap) included:")
    display(df_features.head(3))

    # --- THE CURE ---
    df_features = df_features.drop(columns=['LEAK_clicks'])
    print("\nTrap sprung and removed. Honest feature frame ready.")
    display(df_features.head(3))
except Exception as e:
    print(f"Setup failed: {e}")

CRITICAL: Feature frame with LEAKAGE (The Trap) included:


,date,client_id,content_id,device,position,day_of_week,LEAK_clicks
0,2025-03-01,client_73cda7b4e4f265ea,content_30744f0df03fae4d,unknown,65.5,5,0
1,2025-03-01,client_73cda7b4e4f265ea,content_eb59ee4f00886795,unknown,74.0,5,0
2,2025-03-01,client_73cda7b4e4f265ea,content_e487f64cb6f0bf09,unknown,72.0,5,0



Trap sprung and removed. Honest feature frame ready.


,date,client_id,content_id,device,position,day_of_week
0,2025-03-01,client_73cda7b4e4f265ea,content_30744f0df03fae4d,unknown,65.5,5
1,2025-03-01,client_73cda7b4e4f265ea,content_eb59ee4f00886795,unknown,74.0,5
2,2025-03-01,client_73cda7b4e4f265ea,content_e487f64cb6f0bf09,unknown,72.0,5


In [17]:
try:
    # Re-verify all three facts for the contract summary
    grain_key = ['date', 'client_id', 'content_id', 'device']
    max_dupes = df_sample.groupby(grain_key).size().max()
    print(f"Fact 1 - Grain is unique: {max_dupes == 1}")

    d_min, d_max = df_sample['date'].min(), df_sample['date'].max()
    print(f"Fact 2 - Row count: {len(df_sample):,} | Span: {d_min} to {d_max}")

    survivors = df_sample[df_sample['is_valid_signal'] == True]
    print(f"Fact 3 - Availability survival: {len(survivors):,} rows ({(len(survivors)/len(df_sample)):.1%})")
except NameError:
    print("Run the setup cell successfully first.")

Fact 1 - Grain is unique: True
Fact 2 - Row count: 20,000 | Span: 2025-03-01 to 2025-03-21
Fact 3 - Availability survival: 19,966 rows (99.8%)


## 4. Data limits

**Named Limitation:** This slice only captures users who actually reached the search results page where our domain appeared (GSC data); it cannot tell us about 'Zero-click' searches or keywords where we have zero visibility.

In [23]:
# 4. Data Limits: Checking for overlaps or imbalance
try:
    # Updated to check concentration of content_id instead of landing_page
    top_pages = df_sample['content_id'].value_counts(normalize=True).head(5)
    print("Top 5 content IDs (concentration check):")
    print(top_pages)
except Exception as e:
    print(f"Check failed: {e}")

Top 5 content IDs (concentration check):
content_id
content_e671e2d045c1c4b7    0.00025
content_17671d72889f5eeb    0.00025
content_de2a6f4a19848e96    0.00025
content_bb007c2563ba397c    0.00025
content_a7a62123de55f47b    0.00025
Name: proportion, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.